In [8]:
import numpy as np
from pathlib import Path
from collections import Counter
from parsing_data import parse_dataset
import yaml


PROJECT_ROOT = Path.cwd().parent.parent

In [9]:
# --- ВЫЧИСЛЕНИЕ RARITYWEIGHT ---
train_data = parse_dataset(PROJECT_ROOT / 'data' / 'ruccod_train')  # путь к обучающей выборке

code_counter = Counter()
for item in train_data:
    code_counter.update(item['codes'])

print(f'Всего документов: {len(train_data)}')
print(f'Уникальных кодов МКБ-10: {len(code_counter)}')
print('Топ-10 по частоте:')
for code, freq in code_counter.most_common(10):
    print(f'  {code}: {freq}')

Всего документов: 3000
Уникальных кодов МКБ-10: 1433
Топ-10 по частоте:
  J00: 254
  M42.1: 176
  M79.1: 133
  N72: 121
  M54.2: 99
  K29: 88
  J06.8: 82
  N76: 80
  D25: 78
  J35.0: 74


In [10]:
code_counter

Counter({'J00': 254,
         'M42.1': 176,
         'M79.1': 133,
         'N72': 121,
         'M54.2': 99,
         'K29': 88,
         'J06.8': 82,
         'N76': 80,
         'D25': 78,
         'J35.0': 74,
         'N86': 72,
         'M53.9': 72,
         'Z34': 69,
         'B97.7': 69,
         'A63.8': 64,
         'I11': 63,
         'J02': 61,
         'N80.0': 60,
         'N70': 60,
         'K82.8': 58,
         'J20': 57,
         'M54.4': 56,
         'J04.1': 54,
         'M53.8': 52,
         'I10': 51,
         'K86.1': 51,
         'H52.1': 46,
         'N70.1': 45,
         'K29.5': 44,
         'J34.2': 44,
         'K83.8': 43,
         'R19.8': 42,
         'K29.3': 41,
         'G90.8': 40,
         'M54': 38,
         'N41.1': 38,
         'K29.9': 37,
         'M54.6': 37,
         'J31.2': 35,
         'K58': 35,
         'E03.9': 34,
         'F48.0': 33,
         'J01': 33,
         'E28': 32,
         'K63': 32,
         'M54.8': 32,
         'I83': 31

In [11]:
# Преобразование частот в веса (логарифмическое)
epsilon = 1.0
freqs = np.array(list(code_counter.values()))
log_freq = np.log(freqs + epsilon)
rarity_raw = 1.0 / (log_freq + epsilon)
rarity_norm = (rarity_raw - rarity_raw.min()) / (rarity_raw.max() - rarity_raw.min())

rarity_weights = dict(zip(code_counter.keys(), rarity_norm))

# Для отсутствующих в обучающей выборке кодов — максимальная редкость (1.0)
def get_rarity_weight(code):
    return rarity_weights.get(code, 1.0)

In [13]:
# --- ИНТЕГРАЦИЯ С SEVERITYWEIGHT ---
with open(PROJECT_ROOT / 'data' / 'mkb2descr.yaml', 'r', encoding='utf-8') as f:
    mkb_tree = yaml.safe_load(f)

with open(PROJECT_ROOT / 'src' / 'data' / 'severity_weights.yaml', 'r', encoding='utf-8') as f:
    severity_weights = yaml.safe_load(f)

alpha = 0.6
hybrid_weights = {}

for code in mkb_tree:
    if not isinstance(code, str) or len(code) < 2:
        continue
    sev = severity_weights.get(code, 0.5)     # fallback
    rar = get_rarity_weight(code)
    hybrid_weights[code] = alpha * sev + (1 - alpha) * rar

# hybrid_weights готов для использования в PyTorch Loss

In [18]:
hybrid_weights

{'00': 0.7,
 'A00': 0.8200000000000001,
 'A00-A09': 0.8200000000000001,
 'A00-B99': 0.8200000000000001,
 'A00.0': 0.8200000000000001,
 'A00.1': 0.8200000000000001,
 'A00.9': 0.8200000000000001,
 'A01': 0.76,
 'A01.0': 0.76,
 'A01.1': 0.76,
 'A01.2': 0.76,
 'A01.3': 0.76,
 'A01.4': 0.76,
 'A02': 0.76,
 'A02.0': 0.76,
 'A02.1': 0.76,
 'A02.2': 0.76,
 'A02.8': 0.76,
 'A02.9': 0.603234659823713,
 'A03': 0.76,
 'A03.0': 0.76,
 'A03.1': 0.76,
 'A03.2': 0.76,
 'A03.3': 0.76,
 'A03.8': 0.76,
 'A03.9': 0.76,
 'A04': 0.76,
 'A04.0': 0.76,
 'A04.1': 0.76,
 'A04.2': 0.76,
 'A04.3': 0.76,
 'A04.4': 0.76,
 'A04.5': 0.76,
 'A04.6': 0.76,
 'A04.7': 0.76,
 'A04.8': 0.76,
 'A04.9': 0.76,
 'A05': 0.8200000000000001,
 'A05.0': 0.8200000000000001,
 'A05.1': 0.8200000000000001,
 'A05.2': 0.8200000000000001,
 'A05.3': 0.8200000000000001,
 'A05.4': 0.8200000000000001,
 'A05.8': 0.6304887793296523,
 'A05.9': 0.6304887793296523,
 'A06': 0.8200000000000001,
 'A06.0': 0.8200000000000001,
 'A06.1': 0.8200000000000